In [1]:
# Import libraries
from pathlib import Path
import pandas as pd

# Define project directories
PROJECT_ROOT = Path.cwd().parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DOCS_DIR = PROJECT_ROOT / "documentation"

# Load ELOG datasets from Notebook 01
elog_all = pd.read_csv(
    INTERIM_DIR / "elog_county_election_results.csv"
)

# Load official IEBC datasets from Notebook 02
iebc_2013 = pd.read_csv(
    INTERIM_DIR / "iebc_2013_county_official.csv"
)

iebc_aug_2017 = pd.read_csv(
    INTERIM_DIR / "iebc_2017_august_county_official.csv"
)

iebc_oct_2017 = pd.read_csv(
    INTERIM_DIR / "iebc_2017_october_county_official.csv"
)

# Load reconciliation outputs
recon_2013 = pd.read_csv(
    DOCS_DIR / "reconciliation_2013.csv"
)

recon_aug_2017 = pd.read_csv(
    DOCS_DIR / "reconciliation_2017_august.csv"
)

recon_oct_2017 = pd.read_csv(
    DOCS_DIR / "reconciliation_2017_october.csv"
)

reconciliation_summary = pd.read_csv(
    DOCS_DIR / "notebook02_reconciliation_summary.csv"
)

print("All saved datasets loaded successfully.")

All saved datasets loaded successfully.


In [3]:
# Collect datasets for structural inspection
datasets = {
    "ELOG All Elections": elog_all,
    "IEBC 2013": iebc_2013,
    "IEBC August 2017": iebc_aug_2017,
    "IEBC October 2017": iebc_oct_2017,
    "Reconciliation 2013": recon_2013,
    "Reconciliation August 2017": recon_aug_2017,
    "Reconciliation October 2017": recon_oct_2017,
    "Reconciliation Summary": reconciliation_summary
}

# Display row counts, column counts and missing values
dataset_overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum())
    }
    for name, df in datasets.items()
])

display(dataset_overview)

# Review the reconciliation summary before integration
display(reconciliation_summary)

# Inspect column names to plan safe joins
for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

,dataset,rows,columns,missing_values
0,ELOG All Elections,188,16,0
1,IEBC 2013,47,8,0
2,IEBC August 2017,47,5,0
3,IEBC October 2017,47,4,0
4,Reconciliation 2013,47,29,0
5,Reconciliation August 2017,47,26,0
6,Reconciliation October 2017,47,25,0
7,Reconciliation Summary,4,5,1


,election_id,county_records_expected,county_records_official,official_source,reconciliation_level
0,2013_GENERAL,47,47.0,IEBC 2013 General Election Data,County
1,2017_AUG_GENERAL,47,47.0,IEBC Data Report of 2017 Elections,County
2,2017_OCT_FRESH,47,47.0,IEBC Data Report of 2017 Elections,County
3,2022_GENERAL,47,NaN,IEBC 2022 Post Election Evaluation,National only



ELOG All Elections:
['election_id', 'election_year', 'election_date', 'election_type', 'county', 'registered_voters', 'valid_votes', 'turnout_pct', 'winner', 'winner_votes', 'runner_up', 'margin', 'source', 'source_file', 'valid_vote_rate_pct', 'turnout_difference_pp']

IEBC 2013:
['county_no', 'county_raw', 'registered_biometric', 'registered_non_biometric', 'registered_total', 'number_voted', 'official_turnout_pct', 'county_key']

IEBC August 2017:
['county_code', 'registered_voters_official', 'valid_votes_official', 'rejected_ballots_official', 'county_key']

IEBC October 2017:
['county_key', 'registered_voters_official', 'valid_votes_official', 'rejected_ballots_official']

Reconciliation 2013:
['election_id', 'election_year', 'election_date', 'election_type', 'county', 'registered_voters', 'valid_votes', 'turnout_pct', 'winner', 'winner_votes', 'runner_up', 'margin', 'source', 'source_file', 'valid_vote_rate_pct', 'turnout_difference_pp', 'county_key', 'county_no', 'county_raw', 

In [5]:
# Summarise registration differences for 2013
summary_2013 = {
    "election": "2013 General",
    "counties": len(recon_2013),
    "registration_matches": int(
        (recon_2013["diff_vs_biometric"] == 0).sum()
    ),
    "registration_differences": int(
        (recon_2013["diff_vs_biometric"] != 0).sum()
    ),
    "maximum_absolute_difference": int(
        recon_2013["diff_vs_biometric"].abs().max()
    )
}

# Summarise registration and valid-vote differences for 2017
summary_2017 = []

for election_name, df in [
    ("August 2017", recon_aug_2017),
    ("October 2017", recon_oct_2017)
]:

    summary_2017.append({
        "election": election_name,
        "counties": len(df),

        "registration_matches": int(
            (df["registered_difference"] == 0).sum()
        ),

        "registration_differences": int(
            (df["registered_difference"] != 0).sum()
        ),

        "maximum_absolute_difference": int(
            df["registered_difference"].abs().max()
        ),

        "valid_vote_matches": int(
            (df["valid_votes_difference"] == 0).sum()
        ),

        "valid_vote_differences": int(
            (df["valid_votes_difference"] != 0).sum()
        ),

        "maximum_valid_vote_difference": int(
            df["valid_votes_difference"].abs().max()
        )
    })

# Combine reconciliation summaries
reconciliation_assessment = pd.DataFrame(
    [summary_2013] + summary_2017
)

display(reconciliation_assessment)

,election,counties,registration_matches,registration_differences,maximum_absolute_difference,valid_vote_matches,valid_vote_differences,maximum_valid_vote_difference
0,2013 General,47,42,5,12,NaN,NaN,NaN
1,August 2017,47,47,0,0,45.0,2.0,60516.0
2,October 2017,47,47,0,0,47.0,0.0,0.0


In [7]:
# Investigate registration and valid-vote discrepancies for both 2017 elections

for election_name, df in [
    ("August 2017", recon_aug_2017),
    ("October 2017", recon_oct_2017)
]:

    print(f"\n{'=' * 65}")
    print(f"{election_name.upper()} — RECONCILIATION REVIEW")
    print(f"{'=' * 65}")

    # Compare county-level registration totals across sources
    print("\nRegistration totals:")
    print(
        df[
            ["registered_voters", "registered_voters_official"]
        ].sum().to_string()
    )

    # Compare county-level valid-vote totals across sources
    print("\nValid-vote totals:")
    print(
        df[
            ["valid_votes", "valid_votes_official"]
        ].sum().to_string()
    )

    # Identify the 10 counties with the largest registration discrepancies
    print("\nTop 10 registration discrepancies:")
    display(
        df.assign(
            abs_registration_difference=(
                df["registered_difference"].abs()
            )
        )
        .sort_values(
            "abs_registration_difference",
            ascending=False
        )
        [
            [
                "county",
                "registered_voters",
                "registered_voters_official",
                "registered_difference"
            ]
        ]
        .head(10)
    )

    # Identify the 10 counties with the largest valid-vote discrepancies
    print("\nTop 10 valid-vote discrepancies:")
    display(
        df.assign(
            abs_valid_vote_difference=(
                df["valid_votes_difference"].abs()
            )
        )
        .sort_values(
            "abs_valid_vote_difference",
            ascending=False
        )
        [
            [
                "county",
                "valid_votes",
                "valid_votes_official",
                "valid_votes_difference"
            ]
        ]
        .head(10)
    )


AUGUST 2017 — RECONCILIATION REVIEW

Registration totals:
registered_voters             19607064
registered_voters_official    19607064

Valid-vote totals:
valid_votes             15111730
valid_votes_official    15142702

Top 10 registration discrepancies:


,county,registered_voters,registered_voters_official,registered_difference
0,Mombasa,580644,580644,0
35,Bomet,322024,322024,0
26,Uasin Gishu,450159,450159,0
27,Elgeyo-Marakwet,180679,180679,0
28,Nandi,346102,346102,0
29,Baringo,232311,232311,0
30,Laikipia,246693,246693,0
31,Nakuru,949971,949971,0
32,Narok,341761,341761,0
33,Kajiado,411267,411267,0



Top 10 valid-vote discrepancies:


,county,valid_votes,valid_votes_official,valid_votes_difference
41,Kisumu,378903,439419,-60516
22,Turkana,160912,131368,29544
0,Mombasa,342337,342337,0
34,Kericho,294215,294215,0
26,Uasin Gishu,341575,341575,0
27,Elgeyo-Marakwet,146507,146507,0
28,Nandi,270895,270895,0
29,Baringo,190411,190411,0
30,Laikipia,199502,199502,0
31,Nakuru,754405,754405,0



OCTOBER 2017 — RECONCILIATION REVIEW

Registration totals:
registered_voters             19607030
registered_voters_official    19607030

Valid-vote totals:
valid_votes             7614866
valid_votes_official    7614866

Top 10 registration discrepancies:


,county,registered_voters,registered_voters_official,registered_difference
0,Mombasa,580613,580613,0
35,Bomet,322024,322024,0
26,Uasin Gishu,450159,450159,0
27,Elgeyo-Marakwet,180679,180679,0
28,Nandi,346102,346102,0
29,Baringo,232311,232311,0
30,Laikipia,246693,246693,0
31,Nakuru,949971,949971,0
32,Narok,341761,341761,0
33,Kajiado,411267,411267,0



Top 10 valid-vote discrepancies:


,county,valid_votes,valid_votes_official,valid_votes_difference
0,Mombasa,80464,80464,0
35,Bomet,226746,226746,0
26,Uasin Gishu,267370,267370,0
27,Elgeyo-Marakwet,127623,127623,0
28,Nandi,230589,230589,0
29,Baringo,152318,152318,0
30,Laikipia,173123,173123,0
31,Nakuru,651017,651017,0
32,Narok,128996,128996,0
33,Kajiado,171359,171359,0


***These discrepancies are substantial enough to investigate, but the fact that 45 counties match exactly suggests that we may be dealing with isolated source or extraction issues rather than a widespread inconsistency.***
### Our next step: Check the original August 2017 PDF

In [11]:
# Import PDF reader
from pypdf import PdfReader

# Locate original IEBC 2017 report
IEBC_2017_FILE = (
    PROJECT_ROOT
    / "data"
    / "Raw"
    / "reference"
    / "IEBC_2017_Presidential_Declaration_Results.pdf"
)

reader = PdfReader(IEBC_2017_FILE)

# Inspect the two PDF pages containing the August county table
for page_index in [17, 18]:

    print(f"\n{'=' * 70}")
    print(f"PDF PAGE {page_index + 1}")
    print(f"{'=' * 70}")

    page_text = reader.pages[page_index].extract_text() or ""
    lines = page_text.splitlines()

    # Find the original rows for the two counties
    for i, line in enumerate(lines):

        if "TURKANA" in line.upper() or "KISUMU" in line.upper():

            print("\nCounty found:", line.strip())

            # Show neighbouring lines to identify wrapped rows
            start = max(0, i - 1)
            end = min(len(lines), i + 3)

            for nearby_line in lines[start:end]:
                print(repr(nearby_line))


PDF PAGE 18

County found: 23 TURKANA 191,435 643 91 79 290 58744 79 379 71063 131368 780
'22 KIAMBU 1,181,076 761 344 154 1,028 912,588 320 767 69,190 985,152 3,828 '
'23 TURKANA 191,435 643 91 79 290 58744 79 379 71063 131368 780 '
'24 WEST POKOT 180,241 288 114 94 269 97,620 99 482 52,120 151,086 448 '
'25 SAMBURU 82,794 251 33 25 74 31,746 19 189 31,615 63,952 70 '

PDF PAGE 19

County found: 42 KISUMU 539,593 184 140 69 90 7,411 39 1,007 369,963 439419 1,127
'41 SIAYA 457,957 146 54 34 61 2,494 46 533 375,712 379,080 1,105 '
'42 KISUMU 539,593 184 140 69 90 7,411 39 1,007 369,963 439419 1,127 '
'43 HOMA BAY 476,932 94 58 23 20 1,960 39 453 400,351 402,998 1,017 '
'44 MIGORI 388,700 283 148 151 126 46,112 158 1,122 274,161 322,261 1,148 '


***This is a significant finding. Cell 5 confirms that our August 2017 PDF extraction captured the values correctly for both Turkana and Kisumu. The discrepancies are therefore not caused by our parser selecting the wrong columns.***

*Let's create a small audit table in memory, documenting these discrepancies and their current status.*

In [15]:
# Document the two unresolved August 2017 valid-vote discrepancies
august_2017_audit = pd.DataFrame([
    {
        "election_id": "2017_AUG_GENERAL",
        "county": "Turkana",
        "elog_valid_votes": 160912,
        "iebc_reported_valid_votes": 131368,
        "candidate_votes_sum": 131368,
        "difference_elog_minus_iebc": 29544,
        "finding": "IEBC candidate sum matches printed total; ELOG differs",
        "status": "UNRESOLVED_SOURCE_DIFFERENCE"
    },
    {
        "election_id": "2017_AUG_GENERAL",
        "county": "Kisumu",
        "elog_valid_votes": 378903,
        "iebc_reported_valid_votes": 439419,
        "candidate_votes_sum": 378903,
        "difference_elog_minus_iebc": -60516,
        "finding": "Candidate sum matches ELOG; IEBC printed total differs",
        "status": "INTERNAL_IEBC_TOTAL_INCONSISTENCY"
    }
])

# Review the audit findings
display(august_2017_audit)

,election_id,county,elog_valid_votes,iebc_reported_valid_votes,candidate_votes_sum,difference_elog_minus_iebc,finding,status
0,2017_AUG_GENERAL,Turkana,160912,131368,131368,29544,IEBC candidate sum matches printed total; ELOG...,UNRESOLVED_SOURCE_DIFFERENCE
1,2017_AUG_GENERAL,Kisumu,378903,439419,378903,-60516,Candidate sum matches ELOG; IEBC printed total...,INTERNAL_IEBC_TOTAL_INCONSISTENCY


## Turkana
The eight candidate figures add up to **131,368**, confirming that the printed IEBC total is internally consistent.

However, ELOG reports **160,912** valid votes, a difference of **29,544**.

***Finding:*** An unresolved discrepancy between the two sources.

## Kisumu
However, adding the eight candidate figures produces **378,903**, exactly matching ELOG.

The printed total exceeds the candidate sum by **60,516** votes.

***Finding:*** An internal inconsistency in the IEBC report's Kisumu row.

For our eventual analytical dataset:
- Kisumu: Use 378,903 as the provisional valid-vote figure, supported by the candidate sum and ELOG.
- Turkana: Keep both figures flagged as unresolved until we can identify additional supporting evidence.
- All other counties: Retain their reconciled figures, subject to our remaining national-total checks.
We should also avoid using the extracted IEBC national total as an automatic validation of every county row. The report's printed national total may not equal the sum of its county totals when a county contains an internal inconsistency.

## National-total reconciliation
The IEBC report provides the following August 2017 national figures:
- **Measure**	                              - **Official IEBC figure**
- Registered voters, including diaspora	    19,611,423
- Valid votes, including diaspora	        15,114,622
- Diaspora registered voters	                 4,393
- Diaspora valid votes	                         2,844
- Expected county-only registered voters	19,607,030
- Expected county-only valid votes	        15,111,778

Our earlier results showed that ELOG and the extracted IEBC county registration totals both equal 19,607,064, which is 34 higher than the national benchmark after excluding diaspora.

We should investigate that difference alongside the valid-vote discrepancies.


In [33]:
# Cell 7: Reconcile August 2017 county totals
# against the IEBC published national totals

# Official national figures from IEBC 2017 report
official_national = {
    "registered_voters": 19611423,
    "valid_votes": 15114622
}

# Diaspora figures from the same report
diaspora = {
    "registered_voters": 4393,
    "valid_votes": 2844
}

# Calculate expected totals for Kenya's 47 counties
expected_county = {
    key: official_national[key] - diaspora[key]
    for key in official_national
}

# Calculate actual totals from our reconciled dataset
comparison = pd.DataFrame([
    {
        "measure": "Registered voters",
        "official_county_expected": expected_county["registered_voters"],
        "elog_county_total": recon_aug_2017["registered_voters"].sum(),
        "iebc_extracted_county_total": recon_aug_2017[
            "registered_voters_official"
        ].sum()
    },
    {
        "measure": "Valid votes",
        "official_county_expected": expected_county["valid_votes"],
        "elog_county_total": recon_aug_2017["valid_votes"].sum(),
        "iebc_extracted_county_total": recon_aug_2017[
            "valid_votes_official"
        ].sum()
    }
])

# Calculate differences against the official county benchmark
comparison["elog_difference"] = (
    comparison["elog_county_total"]
    - comparison["official_county_expected"]
)

comparison["iebc_difference"] = (
    comparison["iebc_extracted_county_total"]
    - comparison["official_county_expected"]
)

display(comparison)

,measure,official_county_expected,elog_county_total,iebc_extracted_county_total,elog_difference,iebc_difference
0,Registered voters,19607030,19607064,19607064,34,34
1,Valid votes,15111778,15111730,15142702,-48,30924


Registration: ELOG and the IEBC county extract agree exactly across all 47 counties. However, their combined total exceeds the published national figure (excluding diaspora) by 34 voters. This points to a discrepancy within the published totals rather than between our two datasets.

Valid votes: ELOG is only 48 votes below the national benchmark. The extracted IEBC county totals are 30,924 votes above it.

## What we can conclude
- **Kisumu:** The candidate votes sum to 378,903, matching ELOG rather than the IEBC printed total of 439,419. We have strong evidence of an internal inconsistency in the published IEBC row.
- **Turkana:** The IEBC candidate votes sum to 131,368, matching its printed total. ELOG reports 160,912. We cannot yet determine which is appropriate for our analytical dataset.
- National totals: The 34-voter registration difference and 48-vote ELOG valid-vote shortfall remain unexplained.

**Check whether candidate totals explain the national discrepancy**

In [38]:
# Cell 8: Independently validate the national candidate totals
# from the August 2017 IEBC report (page 19)

candidate_votes = {
    "John Ekuru Aukot": 27311,
    "Mohamed Abduba Dida": 38093,
    "Shakhalaga Khwa Jirongo": 11705,
    "Japheth Kavinga Kaluyu": 16482,
    "Uhuru Kenyatta": 8203290,
    "Michael Wainaina Mwaura": 13257,
    "Joseph William Nyagah": 42259,
    "Raila Odinga": 6762224
}

# Published national valid-vote total
published_national_valid_votes = 15114622

# Independently sum candidate votes
candidate_total = sum(candidate_votes.values())

print("Sum of national candidate votes:", f"{candidate_total:,}")
print("Published national valid votes:", f"{published_national_valid_votes:,}")

print(
    "Difference:",
    f"{candidate_total - published_national_valid_votes:,}"
)

Sum of national candidate votes: 15,114,621
Published national valid votes: 15,114,622
Difference: -1


## 1. What this tells us
We now have four documented reconciliation findings:
| Finding	                                      | Difference	    | Status| 
|--|--|--|
| Kisumu printed valid votes versus candidate sum	 | - 60,516	      | Internal IEBC inconsistency| 
| Turkana ELOG versus IEBC valid votes	             | - 29,544	      | Unresolved source difference| 
| County registration sum versus national benchmark	   | +34	      | Unresolved| 
| National candidate sum versus printed valid votes	     | −1	      | Internal IEBC inconsistency| 

The one-vote difference is negligible for aggregate participation analysis, but it matters for our data-quality documentation.

We should not change the national total to 15,114,621. Instead, retain the published value and record the discrepancy.

## Check the Turkana and Kisumu candidate totals
Before introducing any more sources, let's document the candidate-level evidence programmatically.

In [48]:
# Cell 9: Verify candidate-level totals for the two disputed counties
# Values transcribed from the IEBC August 2017 report, pages 18–19

county_candidates = {
    "Turkana": [643, 91, 79, 290, 58744, 79, 379, 71063],
    "Kisumu": [184, 140, 69, 90, 7411, 39, 1007, 369963]
}

reported_totals = {
    "Turkana": 131368,
    "Kisumu": 439419
}

elog_totals = {
    "Turkana": 160912,
    "Kisumu": 378903
}

results = []

for county, votes in county_candidates.items():

    calculated_total = sum(votes)

    results.append({
        "county": county,
        "candidate_sum": calculated_total,
        "iebc_printed_total": reported_totals[county],
        "elog_total": elog_totals[county],
        "candidate_minus_iebc": (
            calculated_total - reported_totals[county]
        ),
        "candidate_minus_elog": (
            calculated_total - elog_totals[county]
        )
    })

candidate_validation = pd.DataFrame(results)

display(candidate_validation)

,county,candidate_sum,iebc_printed_total,elog_total,candidate_minus_iebc,candidate_minus_elog
0,Turkana,131368,131368,160912,0,-29544
1,Kisumu,378903,439419,378903,-60516,0


**This establishes two different data-quality issues.**

- Kisumu: The IEBC printed total is inconsistent with the sum of its eight candidate votes. We have sufficient evidence to use 378,903 as the provisional analytical value, while retaining the original IEBC figure for audit purposes.

- Turkana: The IEBC printed total agrees with the candidate sum, but differs from ELOG by 29,544 votes. We should not automatically select either value without further verification.

The 34-voter registration discrepancy and 48-vote national valid-vote shortfall also remain documented exceptions.

## Create the August 2017 validation register

In [52]:
# Cell 10: Create a county-level validation register for August 2017
# Preserve original ELOG and IEBC figures for audit purposes

august_validation = recon_aug_2017[
    [
        "election_id",
        "county",
        "registered_voters",
        "registered_voters_official",
        "valid_votes",
        "valid_votes_official",
        "registered_difference",
        "valid_votes_difference"
    ]
].copy()

# Rename source-specific fields for clarity
august_validation = august_validation.rename(columns={
    "registered_voters": "elog_registered_voters",
    "registered_voters_official": "iebc_registered_voters",
    "valid_votes": "elog_valid_votes",
    "valid_votes_official": "iebc_valid_votes"
})

# Assign initial validation status
august_validation["validation_status"] = "MATCHED"

# Flag counties with conflicting registration figures
registration_conflict = (
    august_validation["registered_difference"] != 0
)

# Flag counties with conflicting valid-vote figures
valid_vote_conflict = (
    august_validation["valid_votes_difference"] != 0
)

august_validation.loc[
    registration_conflict | valid_vote_conflict,
    "validation_status"
] = "UNRESOLVED_SOURCE_DIFFERENCE"

# Record the separately verified Kisumu inconsistency
august_validation.loc[
    august_validation["county"] == "Kisumu",
    "validation_status"
] = "INTERNAL_SOURCE_INCONSISTENCY"

# Include the independent candidate-sum evidence
august_validation["candidate_votes_sum"] = pd.NA

august_validation.loc[
    august_validation["county"] == "Kisumu",
    "candidate_votes_sum"
] = 378903

august_validation.loc[
    august_validation["county"] == "Turkana",
    "candidate_votes_sum"
] = 131368

# Review validation status distribution
print("August 2017 validation summary:")
display(
    august_validation["validation_status"]
    .value_counts()
    .rename_axis("validation_status")
    .reset_index(name="county_count")
)

# Review flagged counties
print("\nCounties requiring documented treatment:")
display(
    august_validation.loc[
        august_validation["validation_status"] != "MATCHED"
    ]
)

August 2017 validation summary:


,validation_status,county_count
0,MATCHED,45
1,UNRESOLVED_SOURCE_DIFFERENCE,1
2,INTERNAL_SOURCE_INCONSISTENCY,1



Counties requiring documented treatment:


,election_id,county,elog_registered_voters,iebc_registered_voters,elog_valid_votes,iebc_valid_votes,registered_difference,valid_votes_difference,validation_status,candidate_votes_sum
22,2017_AUG_GENERAL,Turkana,191435,191435,160912,131368,0,29544,UNRESOLVED_SOURCE_DIFFERENCE,131368
41,2017_AUG_GENERAL,Kisumu,539593,539593,378903,439419,0,-60516,INTERNAL_SOURCE_INCONSISTENCY,378903


In [54]:
# Cell 11: Save the August 2017 validation register
# Documentation only — no original source files are modified

validation_file = (
    DOCS_DIR / "validation_register_2017_august.csv"
)

# Confirm expected structure before saving
assert len(august_validation) == 47, (
    "Expected 47 county records."
)

assert august_validation["county"].nunique() == 47, (
    "County names must be unique."
)

assert (
    august_validation["validation_status"]
    .value_counts()
    .to_dict()
    == {
        "MATCHED": 45,
        "UNRESOLVED_SOURCE_DIFFERENCE": 1,
        "INTERNAL_SOURCE_INCONSISTENCY": 1
    }
), "Unexpected validation status distribution."

# Save the validated register
august_validation.to_csv(
    validation_file,
    index=False
)

print("Validation register saved successfully.")
print("Location:", validation_file)
print("Records:", len(august_validation))

Validation register saved successfully.
Location: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\validation_register_2017_august.csv
Records: 47



### Proposed Canonical Data Sources and Validation Rules

The following table outlines the proposed data sources for each election event, based on the reconciliation of ELOG and IEBC records.

| Election | Registration Source | Valid-Vote Source | Validation Treatment |
|---|---|---|---|
| 2013 General | IEBC total register | ELOG, pending final checks | Five small biometric registration differences documented |
| August 2017 | IEBC county figures | Reconciled values | Kisumu provisional; Turkana unresolved |
| October 2017 | IEBC county figures | IEBC/ELOG matched values | All 47 counties reconciled |
| 2022 General | ELOG | ELOG | Official county-level verification outstanding |

**Important methodological considerations:**

- **2013:** The IEBC total register includes biometric and non-biometric registrations. Using this denominator may produce different turnout rates from those originally reported by ELOG.
- **August 2017:** Kisumu has an internal inconsistency in the IEBC printed valid-vote total. Turkana has an unresolved difference between ELOG and IEBC. Both require documented treatment.
- **October 2017:** This fresh presidential election must remain a separate electoral event from the August general election.
- **2022:** ELOG county-level figures remain provisionally usable but have not been independently verified against official IEBC county-level records.
- **Turnout methodology:** Where available, turnout should be calculated using total ballots cast (valid votes plus rejected ballots) divided by registered voters, rather than valid votes alone.

**Status:** These are proposed analytical field-selection rules, not finalised corrections. Original source figures will be preserved, and unresolved discrepancies will remain flagged until a documented decision is made.


## Define Canonical Field-Selection Rules

In [58]:

# Cell 12: Define canonical field-selection rules
# These rules govern how source fields will be selected
# when preparing the integrated electoral dataset.
#
# IMPORTANT:
# - Original ELOG and IEBC data remain unchanged.
# - Unresolved differences are not automatically corrected.
# - Turnout and valid-vote participation are separate measures.
# - October 2017 remains a separate election event.


canonical_rules = pd.DataFrame([
    {
        "election_id": "2013_GENERAL",
        "registration_field": "registered_total",
        "registration_source": "IEBC",
        "valid_votes_field": "valid_votes",
        "valid_votes_source": "ELOG_PROVISIONAL",
        "ballots_cast_field": "number_voted",
        "ballots_cast_source": "IEBC",
        "validation_status": "PARTIALLY_RECONCILED",
        "notes": (
            "Use IEBC total registration, including non-biometric "
            "voters. ELOG valid votes remain provisional."
        )
    },
    {
        "election_id": "2017_AUG_GENERAL",
        "registration_field": "registered_voters_official",
        "registration_source": "IEBC",
        "valid_votes_field": "valid_votes_official",
        "valid_votes_source": "IEBC_WITH_EXCEPTIONS",
        "ballots_cast_field": "valid_votes_official + rejected_ballots_official",
        "ballots_cast_source": "IEBC_WITH_EXCEPTIONS",
        "validation_status": "EXCEPTIONS_DOCUMENTED",
        "notes": (
            "45 counties match. Kisumu requires candidate-sum "
            "treatment; Turkana remains unresolved. "
            "Do not automatically select a valid-vote figure "
            "for either exception."
        )
    },
    {
        "election_id": "2017_OCT_FRESH",
        "registration_field": "registered_voters_official",
        "registration_source": "IEBC",
        "valid_votes_field": "valid_votes_official",
        "valid_votes_source": "IEBC",
        "ballots_cast_field": "valid_votes_official + rejected_ballots_official",
        "ballots_cast_source": "IEBC",
        "validation_status": "COUNTY_RECONCILED",
        "notes": (
            "All 47 counties match ELOG for registration "
            "and valid votes. Treat as a separate fresh "
            "presidential election."
        )
    },
    {
        "election_id": "2022_GENERAL",
        "registration_field": "registered_voters",
        "registration_source": "ELOG_PROVISIONAL",
        "valid_votes_field": "valid_votes",
        "valid_votes_source": "ELOG_PROVISIONAL",
        "ballots_cast_field": None,
        "ballots_cast_source": "NOT_VERIFIED",
        "validation_status": "COUNTY_SOURCE_UNVERIFIED",
        "notes": (
            "Official IEBC county-level verification is "
            "outstanding. Do not infer ballots cast from "
            "valid votes or treat valid-vote participation "
            "as official turnout."
        )
    }
])

# Confirm that every election event has one rule
expected_elections = set(elog_all["election_id"].unique())

assert set(canonical_rules["election_id"]) == expected_elections, (
    "Canonical rules do not match the ELOG election events."
)

assert canonical_rules["election_id"].is_unique, (
    "Each election event must have exactly one rule."
)

# Display the proposed canonical rules
print("CANONICAL FIELD-SELECTION RULES")
print("=" * 60)

display(canonical_rules)

# Display the validation status for each election
print("\nVALIDATION STATUS BY ELECTION")

display(
    canonical_rules[
        [
            "election_id",
            "registration_source",
            "valid_votes_source",
            "ballots_cast_source",
            "validation_status"
        ]
    ]
)


CANONICAL FIELD-SELECTION RULES


,election_id,registration_field,registration_source,valid_votes_field,valid_votes_source,ballots_cast_field,ballots_cast_source,validation_status,notes
0,2013_GENERAL,registered_total,IEBC,valid_votes,ELOG_PROVISIONAL,number_voted,IEBC,PARTIALLY_RECONCILED,"Use IEBC total registration, including non-bio..."
1,2017_AUG_GENERAL,registered_voters_official,IEBC,valid_votes_official,IEBC_WITH_EXCEPTIONS,valid_votes_official + rejected_ballots_official,IEBC_WITH_EXCEPTIONS,EXCEPTIONS_DOCUMENTED,45 counties match. Kisumu requires candidate-s...
2,2017_OCT_FRESH,registered_voters_official,IEBC,valid_votes_official,IEBC,valid_votes_official + rejected_ballots_official,IEBC,COUNTY_RECONCILED,All 47 counties match ELOG for registration an...
3,2022_GENERAL,registered_voters,ELOG_PROVISIONAL,valid_votes,ELOG_PROVISIONAL,None,NOT_VERIFIED,COUNTY_SOURCE_UNVERIFIED,Official IEBC county-level verification is out...



VALIDATION STATUS BY ELECTION


,election_id,registration_source,valid_votes_source,ballots_cast_source,validation_status
0,2013_GENERAL,IEBC,ELOG_PROVISIONAL,IEBC,PARTIALLY_RECONCILED
1,2017_AUG_GENERAL,IEBC,IEBC_WITH_EXCEPTIONS,IEBC_WITH_EXCEPTIONS,EXCEPTIONS_DOCUMENTED
2,2017_OCT_FRESH,IEBC,IEBC,IEBC,COUNTY_RECONCILED
3,2022_GENERAL,ELOG_PROVISIONAL,ELOG_PROVISIONAL,NOT_VERIFIED,COUNTY_SOURCE_UNVERIFIED


**Important:** For August 2017, the rule identifies IEBC as the preferred source but does not authorise blindly adopting its printed totals for Kisumu and Turkana. We will handle those two counties explicitly when implementing the rules.

For 2022, we will preserve ELOG's reported turnout percentage as a source-reported measure, but distinguish it from a turnout rate independently recalculated using official ballots cast.

## Build the working dataset

In [62]:
# Cell 13: Build the working analytical dataset
# Keep source values separate from selected analytical values.
# Do not modify the original ELOG or IEBC DataFrames.

working_elections = elog_all.copy()

# Check that every election has 47 counties
assert working_elections.groupby("election_id").size().eq(47).all()
assert len(working_elections) == 188

# Preserve the original ELOG fields under explicit source names
working_elections = working_elections.rename(columns={
    "registered_voters": "elog_registered_voters",
    "valid_votes": "elog_valid_votes",
    "turnout_pct": "elog_reported_turnout_pct"
})

# Attach the proposed source-selection and validation rules
working_elections = working_elections.merge(
    canonical_rules[
        [
            "election_id",
            "registration_source",
            "valid_votes_source",
            "ballots_cast_source",
            "validation_status"
        ]
    ],
    on="election_id",
    how="left",
    validate="many_to_one"
)

# Initialise canonical fields as nullable values
working_elections["analytical_registered_voters"] = pd.NA
working_elections["analytical_valid_votes"] = pd.NA
working_elections["analytical_ballots_cast"] = pd.NA

# Preserve source-level validation at county level
working_elections["county_validation_status"] = pd.NA

# Attach official IEBC registration, valid votes and ballot counts
official_frames = []

for election_id, df in [
    ("2013_GENERAL", recon_2013),
    ("2017_AUG_GENERAL", recon_aug_2017),
    ("2017_OCT_FRESH", recon_oct_2017)
]:
    if election_id == "2013_GENERAL":
        official = df[
            ["county_key", "registered_total", "number_voted"]
        ].copy()

        official = official.rename(columns={
            "registered_total": "iebc_registered_voters",
            "number_voted": "iebc_ballots_cast"
        })

        official["iebc_valid_votes"] = pd.NA
        official["iebc_rejected_ballots"] = pd.NA

    else:
        official = df[
            [
                "county_key",
                "registered_voters_official",
                "valid_votes_official",
                "rejected_ballots_official"
            ]
        ].copy()

        official = official.rename(columns={
            "registered_voters_official": "iebc_registered_voters",
            "valid_votes_official": "iebc_valid_votes",
            "rejected_ballots_official": "iebc_rejected_ballots"
        })

        official["iebc_ballots_cast"] = (
            official["iebc_valid_votes"]
            + official["iebc_rejected_ballots"]
        )

    official["election_id"] = election_id
    official_frames.append(official)

official_combined = pd.concat(
    official_frames,
    ignore_index=True
)

# Create consistent county keys for merging
working_elections["county_key"] = (
    working_elections["county"]
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

# Attach official source figures without overwriting ELOG
working_elections = working_elections.merge(
    official_combined,
    on=["election_id", "county_key"],
    how="left",
    validate="one_to_one",
    indicator="official_merge_status"
)

# Assign 2013 analytical fields
mask_2013 = working_elections["election_id"].eq("2013_GENERAL")

working_elections.loc[
    mask_2013, "analytical_registered_voters"
] = working_elections.loc[mask_2013, "iebc_registered_voters"]

working_elections.loc[
    mask_2013, "analytical_valid_votes"
] = working_elections.loc[mask_2013, "elog_valid_votes"]

working_elections.loc[
    mask_2013, "analytical_ballots_cast"
] = working_elections.loc[mask_2013, "iebc_ballots_cast"]

working_elections.loc[
    mask_2013, "county_validation_status"
] = "PARTIALLY_RECONCILED"

# Assign 2017 analytical fields
mask_2017 = working_elections["election_id"].isin([
    "2017_AUG_GENERAL",
    "2017_OCT_FRESH"
])

working_elections.loc[
    mask_2017, "analytical_registered_voters"
] = working_elections.loc[mask_2017, "iebc_registered_voters"]

working_elections.loc[
    mask_2017, "analytical_valid_votes"
] = working_elections.loc[mask_2017, "iebc_valid_votes"]

working_elections.loc[
    mask_2017, "analytical_ballots_cast"
] = working_elections.loc[mask_2017, "iebc_ballots_cast"]

working_elections.loc[
    mask_2017, "county_validation_status"
] = "MATCHED"

# Apply August 2017 exception statuses
mask_aug = working_elections["election_id"].eq(
    "2017_AUG_GENERAL"
)

for county, status in [
    ("Kisumu", "INTERNAL_SOURCE_INCONSISTENCY"),
    ("Turkana", "UNRESOLVED_SOURCE_DIFFERENCE")
]:
    exception_mask = mask_aug & working_elections["county"].eq(county)

    working_elections.loc[
        exception_mask, "county_validation_status"
    ] = status

    # Do not silently select a disputed valid-vote figure
    working_elections.loc[
        exception_mask, "analytical_valid_votes"
    ] = pd.NA

    working_elections.loc[
        exception_mask, "analytical_ballots_cast"
    ] = pd.NA

# Assign provisional 2022 source fields
mask_2022 = working_elections["election_id"].eq("2022_GENERAL")

working_elections.loc[
    mask_2022, "analytical_registered_voters"
] = working_elections.loc[mask_2022, "elog_registered_voters"]

working_elections.loc[
    mask_2022, "analytical_valid_votes"
] = working_elections.loc[mask_2022, "elog_valid_votes"]

working_elections.loc[
    mask_2022, "county_validation_status"
] = "COUNTY_SOURCE_UNVERIFIED"

# Validate expected structure
assert len(working_elections) == 188
assert working_elections["county_validation_status"].notna().all()

print("Working dataset created:", working_elections.shape)

print("\nCounty validation status:")
display(
    pd.crosstab(
        working_elections["election_id"],
        working_elections["county_validation_status"]
    )
)

print("\nMissing analytical values by election:")
display(
    working_elections.groupby("election_id")[
        [
            "analytical_registered_voters",
            "analytical_valid_votes",
            "analytical_ballots_cast"
        ]
    ].apply(lambda df: df.isna().sum())
)

print("\nAugust 2017 exceptions:")
display(
    working_elections.loc[
        mask_aug & working_elections["county"].isin(
            ["Kisumu", "Turkana"]
        ),
        [
            "county",
            "elog_valid_votes",
            "iebc_valid_votes",
            "analytical_valid_votes",
            "analytical_ballots_cast",
            "county_validation_status"
        ]
    ]
)


Working dataset created: (188, 30)

County validation status:


county_validation_status,COUNTY_SOURCE_UNVERIFIED,INTERNAL_SOURCE_INCONSISTENCY,MATCHED,PARTIALLY_RECONCILED,UNRESOLVED_SOURCE_DIFFERENCE
election_id,,,,,
2013_GENERAL,0,0,0,47,0
2017_AUG_GENERAL,0,1,45,0,1
2017_OCT_FRESH,0,0,47,0,0
2022_GENERAL,47,0,0,0,0



Missing analytical values by election:


,analytical_registered_voters,analytical_valid_votes,analytical_ballots_cast
election_id,,,
2013_GENERAL,9,0,9
2017_AUG_GENERAL,9,11,11
2017_OCT_FRESH,9,9,9
2022_GENERAL,0,0,47



August 2017 exceptions:


,county,elog_valid_votes,iebc_valid_votes,analytical_valid_votes,analytical_ballots_cast,county_validation_status
69,Turkana,160912,131368,<NA>,<NA>,UNRESOLVED_SOURCE_DIFFERENCE
88,Kisumu,378903,439419,<NA>,<NA>,INTERNAL_SOURCE_INCONSISTENCY


Diagnose county matching failures


Before modifying anything, let's identify the nine affected counties.

In [69]:

# Cell 14: Diagnose county-key matching failures
# This is a diagnostic step only; no datasets are modified.

# Identify records that failed to match official IEBC data
unmatched_counties = working_elections.loc[
    (working_elections["election_id"] != "2022_GENERAL")
    & (working_elections["official_merge_status"] != "both"),
    [
        "election_id",
        "county",
        "county_key",
        "official_merge_status"
    ]
].copy()

print("UNMATCHED COUNTY RECORDS")
print("=" * 60)

display(unmatched_counties)

# Count unmatched records by election
print("\nUnmatched records by election:")

display(
    unmatched_counties.groupby("election_id")
    .size()
    .reset_index(name="unmatched_records")
)

# Compare unique county keys across sources
elog_keys = set(
    working_elections.loc[
        working_elections["election_id"] == "2017_AUG_GENERAL",
        "county_key"
    ]
)

iebc_keys = set(
    recon_aug_2017["county_key"]
)

print("\nCounty keys present in ELOG but not IEBC:")
print(sorted(elog_keys - iebc_keys))

print("\nCounty keys present in IEBC but not ELOG:")
print(sorted(iebc_keys - elog_keys))

# Check the number of distinct unmatched county names
print(
    "\nDistinct unmatched counties:",
    unmatched_counties["county"].nunique()
)


UNMATCHED COUNTY RECORDS


,election_id,county,county_key,official_merge_status
3,2013_GENERAL,Tana River,TANARIVER,left_only
5,2013_GENERAL,Taita Taveta,TAITATAVETA,left_only
12,2013_GENERAL,Tharaka-Nithi,THARAKANITHI,left_only
23,2013_GENERAL,West Pokot,WESTPOKOT,left_only
25,2013_GENERAL,Trans Nzoia,TRANSNZOIA,left_only
26,2013_GENERAL,Uasin Gishu,UASINGISHU,left_only
27,2013_GENERAL,Elgeyo-Marakwet,ELGEYOMARAKWET,left_only
42,2013_GENERAL,Homa Bay,HOMABAY,left_only
46,2013_GENERAL,Nairobi City,NAIROBICITY,left_only
50,2017_AUG_GENERAL,Tana River,TANARIVER,left_only



Unmatched records by election:


,election_id,unmatched_records
0,2013_GENERAL,9
1,2017_AUG_GENERAL,9
2,2017_OCT_FRESH,9



County keys present in ELOG but not IEBC:
['ELGEYOMARAKWET', 'HOMABAY', 'NAIROBICITY', 'TAITATAVETA', 'TANARIVER', 'THARAKANITHI', 'TRANSNZOIA', 'UASINGISHU', 'WESTPOKOT']

County keys present in IEBC but not ELOG:
['ELGEYO MARAKWET', 'HOMA BAY', 'NAIROBI CITY', 'TAITA TAVETA', 'TANA RIVER', 'THARAKA NITHI', 'TRANS NZOIA', 'UASIN GISHU', 'WEST POKOT']

Distinct unmatched counties: 9


**Correct county keys and rebuild the merge**
We can repair working_elections directly without rerunning every previous cell. This cell preserves the original source data, replaces the failed merge, and reconstructs the analytical fields using our existing rules.

In [72]:

# Cell 15: Standardise county keys and rebuild the official-data merge
# Original ELOG, IEBC and reconciliation datasets remain unchanged.

# Define one consistent county-key normalisation function
def normalise_county_key(series):
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )


# Rebuild the official source table with standardised keys
official_fixed = official_combined.copy()

official_fixed["county_key"] = normalise_county_key(
    official_fixed["county_key"]
)

# Check that each election/county combination is unique
assert not official_fixed.duplicated(
    ["election_id", "county_key"]
).any(), "Duplicate official county keys detected."


# Remove fields from the previous unsuccessful merge
official_columns = [
    "iebc_registered_voters",
    "iebc_valid_votes",
    "iebc_rejected_ballots",
    "iebc_ballots_cast",
    "official_merge_status"
]

working_fixed = working_elections.drop(
    columns=official_columns
).copy()

# Standardise the ELOG-side keys
working_fixed["county_key"] = normalise_county_key(
    working_fixed["county"]
)

# Merge again using the same key format
working_fixed = working_fixed.merge(
    official_fixed,
    on=["election_id", "county_key"],
    how="left",
    validate="one_to_one",
    indicator="official_merge_status"
)


# Reset analytical fields before applying the rules again
for column in [
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast"
]:
    working_fixed[column] = pd.NA


# 2013: IEBC registration and ballots cast; ELOG valid votes
mask_2013 = working_fixed["election_id"].eq("2013_GENERAL")

working_fixed.loc[
    mask_2013, "analytical_registered_voters"
] = working_fixed.loc[mask_2013, "iebc_registered_voters"]

working_fixed.loc[
    mask_2013, "analytical_valid_votes"
] = working_fixed.loc[mask_2013, "elog_valid_votes"]

working_fixed.loc[
    mask_2013, "analytical_ballots_cast"
] = working_fixed.loc[mask_2013, "iebc_ballots_cast"]


# 2017: IEBC registration, valid votes and ballots cast
mask_2017 = working_fixed["election_id"].isin([
    "2017_AUG_GENERAL",
    "2017_OCT_FRESH"
])

working_fixed.loc[
    mask_2017, "analytical_registered_voters"
] = working_fixed.loc[mask_2017, "iebc_registered_voters"]

working_fixed.loc[
    mask_2017, "analytical_valid_votes"
] = working_fixed.loc[mask_2017, "iebc_valid_votes"]

working_fixed.loc[
    mask_2017, "analytical_ballots_cast"
] = working_fixed.loc[mask_2017, "iebc_ballots_cast"]


# Keep the August 2017 disputed values unresolved
mask_aug_exception = (
    working_fixed["election_id"].eq("2017_AUG_GENERAL")
    & working_fixed["county"].isin(["Kisumu", "Turkana"])
)

working_fixed.loc[
    mask_aug_exception,
    ["analytical_valid_votes", "analytical_ballots_cast"]
] = pd.NA


# 2022: Provisional ELOG figures; ballots cast remain unavailable
mask_2022 = working_fixed["election_id"].eq("2022_GENERAL")

working_fixed.loc[
    mask_2022, "analytical_registered_voters"
] = working_fixed.loc[mask_2022, "elog_registered_voters"]

working_fixed.loc[
    mask_2022, "analytical_valid_votes"
] = working_fixed.loc[mask_2022, "elog_valid_votes"]


# Verify that all 141 official county records now match
official_records = working_fixed.loc[~mask_2022]

assert official_records["official_merge_status"].eq("both").all(), (
    "Some official county records remain unmatched."
)

assert len(working_fixed) == 188, (
    "Unexpected number of election records."
)


# Replace the working dataset only after successful validation
working_elections = working_fixed.copy()


print("CORRECTED WORKING DATASET:", working_elections.shape)

print("\nOfficial merge status:")
display(
    pd.crosstab(
        working_elections["election_id"],
        working_elections["official_merge_status"]
    )
)

print("\nMissing analytical values by election:")
display(
    working_elections.groupby("election_id")[
        [
            "analytical_registered_voters",
            "analytical_valid_votes",
            "analytical_ballots_cast"
        ]
    ].apply(lambda df: df.isna().sum())
)


CORRECTED WORKING DATASET: (188, 30)

Official merge status:


official_merge_status,left_only,both
election_id,,
2013_GENERAL,0,47
2017_AUG_GENERAL,0,47
2017_OCT_FRESH,0,47
2022_GENERAL,47,0



Missing analytical values by election:


,analytical_registered_voters,analytical_valid_votes,analytical_ballots_cast
election_id,,,
2013_GENERAL,0,0,0
2017_AUG_GENERAL,0,2,2
2017_OCT_FRESH,0,0,0
2022_GENERAL,0,0,47


**Why this is an important data-quality lesson**

This is a useful example of why a successful merge operation does not necessarily mean the data has been correctly integrated.

Our initial merge produced 188 records without raising an error, but 27 official records failed to match because of differences in county-key formatting.

By validating missing values and join status, we identified the issue before it could affect the turnout analysis.


### Electoral Participation Measures

Two measures are used to assess electoral participation across Kenya's counties.

**1. Voter Turnout (%)**

$$
\text{Voter Turnout (\%)} =
\frac{\text{Total Ballots Cast}}{\text{Registered Voters}}
\times 100
$$

Voter turnout measures the proportion of registered voters who participated in an election by casting a ballot. Total ballots cast includes both valid votes and rejected ballots.

**2. Valid-Vote Participation (%)**

$$
\text{Valid-Vote Participation (\%)} =
\frac{\text{Valid Votes}}{\text{Registered Voters}}
\times 100
$$

Valid-vote participation measures the proportion of registered voters whose ballots were counted as valid votes.

**Methodological Note**

- **2013:** Turnout is calculated using IEBC's reported number of voters who voted.
- **August 2017:** Turnout is calculated using valid votes plus rejected ballots, excluding the unresolved Kisumu and Turkana records.
- **October 2017:** Turnout is calculated using valid votes plus rejected ballots.
- **2022:** ELOG-reported turnout is retained separately because independently verified county-level ballots-cast figures are not yet available.

These measures must remain distinct to ensure consistent interpretation of electoral participation trends.


In [81]:

# Cell 16: Calculate electoral participation measures
# Preserve the existing working dataset and source values.

participation_df = working_elections.copy()

# Convert analytical fields to numeric values
numeric_columns = [
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast"
]

for column in numeric_columns:
    participation_df[column] = pd.to_numeric(
        participation_df[column],
        errors="raise"
    )

# Ensure every registration denominator is positive
assert participation_df[
    "analytical_registered_voters"
].gt(0).all(), "Invalid registration denominator detected."


# Calculate turnout using ballots cast
participation_df["calculated_turnout_pct"] = (
    participation_df["analytical_ballots_cast"]
    / participation_df["analytical_registered_voters"]
    * 100
).round(2)


# Calculate valid-vote participation separately
participation_df["valid_vote_participation_pct"] = (
    participation_df["analytical_valid_votes"]
    / participation_df["analytical_registered_voters"]
    * 100
).round(2)


# Retain ELOG's reported turnout as a separate measure
participation_df["source_reported_turnout_pct"] = pd.to_numeric(
    participation_df["elog_reported_turnout_pct"],
    errors="raise"
)


# Identify the provenance of the calculated turnout
participation_df["turnout_verification_status"] = (
    "NOT_CALCULATED"
)

participation_df.loc[
    participation_df["election_id"] == "2013_GENERAL",
    "turnout_verification_status"
] = "IEBC_NUMBER_VOTED"

participation_df.loc[
    participation_df["election_id"] == "2017_AUG_GENERAL",
    "turnout_verification_status"
] = "IEBC_BALLOTS_CAST"

participation_df.loc[
    participation_df["election_id"] == "2017_OCT_FRESH",
    "turnout_verification_status"
] = "IEBC_BALLOTS_CAST"

participation_df.loc[
    participation_df["election_id"] == "2022_GENERAL",
    "turnout_verification_status"
] = "SOURCE_REPORTED_ONLY"


# Explicitly flag the August 2017 unresolved turnout figures
exception_mask = (
    participation_df["election_id"].eq("2017_AUG_GENERAL")
    & participation_df["county"].isin(["Kisumu", "Turkana"])
)

participation_df.loc[
    exception_mask,
    "turnout_verification_status"
] = "UNRESOLVED_VALID_VOTES"


# Validate that turnout is not calculated for unresolved records
assert participation_df.loc[
    exception_mask, "calculated_turnout_pct"
].isna().all()

assert participation_df.loc[
    participation_df["election_id"].eq("2022_GENERAL"),
    "calculated_turnout_pct"
].isna().all()


# Summarise available participation measures
print("PARTICIPATION MEASURES — AVAILABILITY")

display(
    participation_df.groupby("election_id")[
        [
            "calculated_turnout_pct",
            "valid_vote_participation_pct",
            "source_reported_turnout_pct"
        ]
    ].count()
)


# Review the two disputed August 2017 counties
print("\nAUGUST 2017 — DISPUTED COUNTIES")

display(
    participation_df.loc[
        exception_mask,
        [
            "county",
            "source_reported_turnout_pct",
            "calculated_turnout_pct",
            "valid_vote_participation_pct",
            "turnout_verification_status"
        ]
    ]
)


# Check for calculated turnout above 100%
print("\nCALCULATED TURNOUT ABOVE 100%:")

display(
    participation_df.loc[
        participation_df["calculated_turnout_pct"] > 100,
        [
            "election_id",
            "county",
            "calculated_turnout_pct"
        ]
    ]
)


PARTICIPATION MEASURES — AVAILABILITY


,calculated_turnout_pct,valid_vote_participation_pct,source_reported_turnout_pct
election_id,,,
2013_GENERAL,47,47,47
2017_AUG_GENERAL,45,45,47
2017_OCT_FRESH,47,47,47
2022_GENERAL,0,47,47



AUGUST 2017 — DISPUTED COUNTIES


,county,source_reported_turnout_pct,calculated_turnout_pct,valid_vote_participation_pct,turnout_verification_status
69,Turkana,68.86,NaN,NaN,UNRESOLVED_VALID_VOTES
88,Kisumu,81.49,NaN,NaN,UNRESOLVED_VALID_VOTES



CALCULATED TURNOUT ABOVE 100%:


,election_id,county,calculated_turnout_pct


The missing values are intentional:
- August 2017: Kisumu and Turkana remain excluded from calculated participation rates because their valid-vote figures are disputed.
- 2022: We retain ELOG-reported turnout, but cannot independently calculate county turnout without verified ballots-cast figures.

## Validate calculated versus reported turnout
This comparison is particularly important because the two measures may use different numerators or registration denominators.

For example, in 2013, we use the IEBC total register, which includes non-biometric registrations. ELOG may have calculated its turnout using the biometric register.

We should therefore examine the differences rather than automatically treating them as errors.

In [87]:

# Cell 17: Compare calculated turnout against ELOG-reported turnout
# Diagnostic only: no source figures or analytical values are modified.

turnout_review = participation_df.copy()

# Calculate the difference in percentage points
turnout_review["turnout_difference_pp_recalculated"] = (
    turnout_review["calculated_turnout_pct"]
    - turnout_review["source_reported_turnout_pct"]
).round(2)

# Summarise differences for counties with calculated turnout
turnout_summary = (
    turnout_review.groupby("election_id")
    .agg(
        counties=("county", "count"),
        comparable_counties=(
            "turnout_difference_pp_recalculated", "count"
        ),
        mean_difference_pp=(
            "turnout_difference_pp_recalculated", "mean"
        ),
        maximum_absolute_difference_pp=(
            "turnout_difference_pp_recalculated",
            lambda s: s.abs().max()
        )
    )
    .reset_index()
)

print("TURNOUT COMPARISON SUMMARY")
display(turnout_summary.round(2))


# Identify the largest county-level differences
comparable = turnout_review.loc[
    turnout_review["turnout_difference_pp_recalculated"].notna()
].copy()

comparable["absolute_difference_pp"] = (
    comparable["turnout_difference_pp_recalculated"].abs()
)

print("\nTOP 15 COUNTY TURNOUT DIFFERENCES")

display(
    comparable.sort_values(
        "absolute_difference_pp",
        ascending=False
    )[
        [
            "election_id",
            "county",
            "source_reported_turnout_pct",
            "calculated_turnout_pct",
            "turnout_difference_pp_recalculated",
            "county_validation_status"
        ]
    ].head(15)
)


# Examine differences greater than one percentage point
material_differences = comparable.loc[
    comparable["absolute_difference_pp"] > 1
]

print("\nCOUNTIES WITH DIFFERENCES ABOVE 1 PERCENTAGE POINT")

display(
    material_differences.groupby("election_id")
    .size()
    .reset_index(name="county_count")
)


TURNOUT COMPARISON SUMMARY


,election_id,counties,comparable_counties,mean_difference_pp,maximum_absolute_difference_pp
0,2013_GENERAL,47,47,-0.22,0.87
1,2017_AUG_GENERAL,47,45,0.24,2.14
2,2017_OCT_FRESH,47,47,0.00,0.00
3,2022_GENERAL,47,0,NaN,NaN



TOP 15 COUNTY TURNOUT DIFFERENCES


,election_id,county,source_reported_turnout_pct,calculated_turnout_pct,turnout_difference_pp_recalculated,county_validation_status
81,2017_AUG_GENERAL,Kericho,80.78,78.64,-2.14,MATCHED
66,2017_AUG_GENERAL,Kirinyaga,86.11,87.15,1.04,MATCHED
22,2013_GENERAL,Turkana,76.22,75.35,-0.87,PARTIALLY_RECONCILED
51,2017_AUG_GENERAL,Lamu,70.96,71.80,0.84,MATCHED
33,2013_GENERAL,Kajiado,87.13,86.39,-0.74,PARTIALLY_RECONCILED
0,2013_GENERAL,Mombasa,66.62,65.93,-0.69,PARTIALLY_RECONCILED
28,2013_GENERAL,Nandi,89.74,89.11,-0.63,PARTIALLY_RECONCILED
61,2017_AUG_GENERAL,Kitui,75.73,76.36,0.63,MATCHED
27,2013_GENERAL,Elgeyo-Marakwet,91.75,91.13,-0.62,PARTIALLY_RECONCILED
48,2017_AUG_GENERAL,Kwale,65.75,66.35,0.60,MATCHED



COUNTIES WITH DIFFERENCES ABOVE 1 PERCENTAGE POINT


,election_id,county_count
0,2017_AUG_GENERAL,2


The Cell 17 results are encouraging. Turnout calculations are broadly consistent with ELOG, but two counties in August 2017 require further investigation: Kericho and Kirinyaga.

**1. Interpretation of Cell 17***

Election	| Counties compared	| Mean difference	| Largest absolute difference	| Assessment
- 2013 General	47	−0.22 pp	0.87 pp	Small differences; denominator definitions may contribute
- August 2017	45	+0.24 pp	2.14 pp	Two counties exceed the review threshold
- October 2017	47	0.00 pp	0.00 pp	Exact agreement at two decimal places
- 2022 General	0	—	—	Independent turnout calculation unavailable

The October 2017 agreement is particularly useful because it supports the consistency of our turnout calculation methodology for that election.

For 2013, the predominantly negative differences are consistent with using the larger IEBC total register as our denominator. However, we have not yet established that registration definitions explain every difference.

### Investigate turnout discrepancies

In [96]:

# Cell 18: Investigate August 2017 turnout discrepancies
# Focus on Kericho and Kirinyaga.
# Corrected version: explicitly convert numeric columns before calculations.
# No original source data is modified.

counties_to_review = ["Kericho", "Kirinyaga"]

turnout_investigation = participation_df.loc[
    participation_df["election_id"].eq("2017_AUG_GENERAL")
    & participation_df["county"].isin(counties_to_review)
].copy()

# Convert the required columns to numeric types
# This prevents Pandas/NumPy rounding errors caused by object dtypes.
numeric_fields = [
    "elog_registered_voters",
    "iebc_registered_voters",
    "elog_valid_votes",
    "iebc_valid_votes",
    "iebc_rejected_ballots",
    "analytical_ballots_cast",
    "source_reported_turnout_pct",
    "calculated_turnout_pct"
]

for column in numeric_fields:
    turnout_investigation[column] = pd.to_numeric(
        turnout_investigation[column],
        errors="raise"
    ).astype("float64")


# Estimate ballots cast implied by ELOG's reported turnout percentage.
# This is approximate because ELOG percentages are rounded.
turnout_investigation["elog_implied_ballots_cast"] = (
    turnout_investigation["source_reported_turnout_pct"]
    / 100
    * turnout_investigation["elog_registered_voters"]
).round(0)


# Compare implied ballots cast against IEBC ballots cast
turnout_investigation["implied_ballot_difference"] = (
    turnout_investigation["elog_implied_ballots_cast"]
    - turnout_investigation["analytical_ballots_cast"]
)


# Independently calculate valid-vote participation
turnout_investigation["valid_vote_rate_check"] = (
    turnout_investigation["iebc_valid_votes"]
    / turnout_investigation["iebc_registered_voters"]
    * 100
).round(2)


# Display the investigation results
print("AUGUST 2017 — TURNOUT DISCREPANCY INVESTIGATION")
print("=" * 65)

display(
    turnout_investigation[
        [
            "county",
            "elog_registered_voters",
            "iebc_registered_voters",
            "elog_valid_votes",
            "iebc_valid_votes",
            "iebc_rejected_ballots",
            "analytical_ballots_cast",
            "source_reported_turnout_pct",
            "calculated_turnout_pct",
            "valid_vote_rate_check",
            "elog_implied_ballots_cast",
            "implied_ballot_difference"
        ]
    ]
)


AUGUST 2017 — TURNOUT DISCREPANCY INVESTIGATION


,county,elog_registered_voters,iebc_registered_voters,elog_valid_votes,iebc_valid_votes,iebc_rejected_ballots,analytical_ballots_cast,source_reported_turnout_pct,calculated_turnout_pct,valid_vote_rate_check,elog_implied_ballots_cast,implied_ballot_difference
66,Kirinyaga,349970.0,349970.0,301856.0,301856.0,3139.0,304995.0,86.11,87.15,86.25,301359.0,-3636.0
81,Kericho,375691.0,375691.0,294215.0,294215.0,1234.0,295449.0,80.78,78.64,78.31,303483.0,8034.0


**Kirinyaga**
ELOG's reported turnout of 86.11% implies approximately 301,359 ballots cast.

However, the IEBC report provides 304,995 ballots cast, comprising 301,856 valid votes and 3,139 rejected ballots.

Interestingly, ELOG's reported turnout is slightly lower than the valid-vote participation rate of 86.25%. That is inconsistent with turnout calculated using the same registered-voter denominator and all ballots cast.


**Kericho**
ELOG's reported turnout of 80.78% implies approximately 303,483 ballots cast.

The IEBC figures give 295,449 ballots cast, a difference of approximately 8,034 ballots.

Unlike Kirinyaga, this difference cannot be explained simply by whether rejected ballots were included.

Our current evidence supports the IEBC-based calculations for both counties, but we should preserve ELOG's reported percentages as separate source values.
**Recommended treatment**
- County	| Analytical turnout	| ELOG turnout	| Proposed status
- Kirinyaga	| 87.15%	| 86.11%	| REPORTED_TURNOUT_DISCREPANCY
- Kericho	| 78.64%	| 80.78%	| REPORTED_TURNOUT_DISCREPANCY

I recommend retaining the IEBC-based turnout calculations because they are reproducible from the documented registration, valid-vote and rejected-ballot figures.

However, we should not describe the ELOG percentages as incorrect without additional evidence about their methodology.

### Document turnout discrepancies
We will create a separate audit register for turnout discrepancies exceeding one percentage point.

This keeps turnout-specific findings distinct from the valid-vote reconciliation exceptions involving Kisumu and Turkana.

In [100]:

# Cell 19: Document material turnout discrepancies
# Use the one-percentage-point diagnostic threshold from Cell 17.
# Do not modify original ELOG or IEBC source data.

TURNOUT_REVIEW_THRESHOLD_PP = 1.0

# Select records where calculated and reported turnout differ
# by more than one percentage point.
turnout_audit = turnout_review.loc[
    turnout_review[
        "turnout_difference_pp_recalculated"
    ].abs() > TURNOUT_REVIEW_THRESHOLD_PP,
    [
        "election_id",
        "county",
        "elog_registered_voters",
        "iebc_registered_voters",
        "elog_valid_votes",
        "iebc_valid_votes",
        "iebc_rejected_ballots",
        "source_reported_turnout_pct",
        "calculated_turnout_pct",
        "turnout_difference_pp_recalculated"
    ]
].copy()

# Assign a separate turnout-specific review status
turnout_audit["turnout_review_status"] = (
    "REPORTED_TURNOUT_DISCREPANCY"
)

# Record the methodological treatment
turnout_audit["analytical_treatment"] = (
    "Retain IEBC-calculated turnout; preserve ELOG reported "
    "turnout separately. Difference remains documented."
)

# Validate expected findings
assert len(turnout_audit) == 2, (
    "Expected two turnout discrepancies above 1 percentage point."
)

assert set(turnout_audit["county"]) == {
    "Kirinyaga", "Kericho"
}, "Unexpected counties identified."

print("TURNOUT DISCREPANCY AUDIT REGISTER")
display(turnout_audit)

print("\nTotal flagged counties:", len(turnout_audit))


TURNOUT DISCREPANCY AUDIT REGISTER


,election_id,county,elog_registered_voters,iebc_registered_voters,elog_valid_votes,iebc_valid_votes,iebc_rejected_ballots,source_reported_turnout_pct,calculated_turnout_pct,turnout_difference_pp_recalculated,turnout_review_status,analytical_treatment
66,2017_AUG_GENERAL,Kirinyaga,349970,349970.0,301856,301856,3139,86.11,87.15,1.04,REPORTED_TURNOUT_DISCREPANCY,Retain IEBC-calculated turnout; preserve ELOG ...
81,2017_AUG_GENERAL,Kericho,375691,375691.0,294215,294215,1234,80.78,78.64,-2.14,REPORTED_TURNOUT_DISCREPANCY,Retain IEBC-calculated turnout; preserve ELOG ...



Total flagged counties: 2


Save the turnout audit register

This cell will save the two findings to documentation/, alongside the August 2017 validation register.

In [103]:

# Cell 20: Save the August 2017 turnout discrepancy audit
# Save documentation only; do not modify source datasets.

turnout_audit_file = (
    DOCS_DIR / "turnout_discrepancy_audit_2017_august.csv"
)

# Validate the audit register before saving
assert len(turnout_audit) == 2, (
    "Expected exactly two turnout discrepancies."
)

assert set(turnout_audit["county"]) == {
    "Kirinyaga", "Kericho"
}, "Unexpected counties in the turnout audit."

assert turnout_audit["election_id"].eq(
    "2017_AUG_GENERAL"
).all(), "Unexpected election event in the audit."

# Save the audit register
turnout_audit.to_csv(
    turnout_audit_file,
    index=False
)

# Verify that the saved file can be read
saved_turnout_audit = pd.read_csv(turnout_audit_file)

assert len(saved_turnout_audit) == len(turnout_audit), (
    "Saved audit record count does not match."
)

print("Turnout audit saved successfully.")
print("File:", turnout_audit_file)
print("Records:", len(saved_turnout_audit))

display(
    saved_turnout_audit[
        [
            "county",
            "source_reported_turnout_pct",
            "calculated_turnout_pct",
            "turnout_difference_pp_recalculated",
            "turnout_review_status"
        ]
    ]
)


Turnout audit saved successfully.
File: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\turnout_discrepancy_audit_2017_august.csv
Records: 2


,county,source_reported_turnout_pct,calculated_turnout_pct,turnout_difference_pp_recalculated,turnout_review_status
0,Kirinyaga,86.11,87.15,1.04,REPORTED_TURNOUT_DISCREPANCY
1,Kericho,80.78,78.64,-2.14,REPORTED_TURNOUT_DISCREPANCY


### Final Data-Quality Assessment
This cell will check:

1. All 188 election-county records are present and unique.

2. All 141 official county records are correctly matched.

3. Missing analytical values occur only where expected.

4. Calculated participation rates fall within 0–100%.

5. Valid votes do not exceed ballots cast.

6. Kisumu and Turkana remain unresolved in August 2017.

7. The 2022 county data retains its unverified status.

In [106]:

# Cell 21: Final data-quality assessment
# Diagnostic only — no source files or working datasets are modified.

quality_df = participation_df.copy()

quality_checks = []

def record_check(check_name, passed, details):
    quality_checks.append({
        "check": check_name,
        "status": "PASS" if passed else "FAIL",
        "details": details
    })


# 1. Verify expected record count
record_check(
    "Expected election records",
    len(quality_df) == 188,
    f"Found {len(quality_df)} records; expected 188."
)


# 2. Verify 47 counties per election
county_counts = quality_df.groupby("election_id")["county"].nunique()

record_check(
    "47 counties per election",
    county_counts.eq(47).all(),
    county_counts.to_dict()
)


# 3. Verify unique election/county combinations
duplicates = quality_df.duplicated(
    ["election_id", "county_key"]
).sum()

record_check(
    "Unique election-county records",
    duplicates == 0,
    f"Duplicate records: {duplicates}"
)


# 4. Verify official county joins
official_mask = quality_df["election_id"].ne("2022_GENERAL")

official_matches = quality_df.loc[
    official_mask, "official_merge_status"
].eq("both").sum()

record_check(
    "Official county reconciliation",
    official_matches == 141,
    f"Matched official county records: {official_matches}/141"
)


# 5. Verify registration completeness
missing_registration = quality_df[
    "analytical_registered_voters"
].isna().sum()

record_check(
    "Registration completeness",
    missing_registration == 0,
    f"Missing registration values: {missing_registration}"
)


# 6. Verify expected missing valid-vote values
missing_valid = quality_df.loc[
    quality_df["analytical_valid_votes"].isna(),
    ["election_id", "county"]
]

expected_missing_valid = {
    ("2017_AUG_GENERAL", "Kisumu"),
    ("2017_AUG_GENERAL", "Turkana")
}

actual_missing_valid = set(
    missing_valid.itertuples(index=False, name=None)
)

record_check(
    "Expected valid-vote exclusions",
    actual_missing_valid == expected_missing_valid,
    f"Missing valid-vote records: {sorted(actual_missing_valid)}"
)


# 7. Verify expected missing ballots-cast values
expected_missing_ballots = (
    expected_missing_valid
    | {
        ("2022_GENERAL", county)
        for county in quality_df.loc[
            quality_df["election_id"].eq("2022_GENERAL"),
            "county"
        ]
    }
)

missing_ballots = quality_df.loc[
    quality_df["analytical_ballots_cast"].isna(),
    ["election_id", "county"]
]

actual_missing_ballots = set(
    missing_ballots.itertuples(index=False, name=None)
)

record_check(
    "Expected ballots-cast exclusions",
    actual_missing_ballots == expected_missing_ballots,
    f"Missing ballots-cast records: {len(actual_missing_ballots)}"
)


# 8. Check calculated participation rates are within 0–100%
turnout_numeric = pd.to_numeric(
    quality_df["calculated_turnout_pct"],
    errors="coerce"
)

valid_rate_numeric = pd.to_numeric(
    quality_df["valid_vote_participation_pct"],
    errors="coerce"
)

rates_valid = (
    turnout_numeric.dropna().between(0, 100).all()
    and valid_rate_numeric.dropna().between(0, 100).all()
)

record_check(
    "Participation rate boundaries",
    rates_valid,
    "All available calculated rates must be between 0% and 100%."
)


# 9. Check valid votes do not exceed ballots cast
valid_votes_numeric = pd.to_numeric(
    quality_df["analytical_valid_votes"],
    errors="coerce"
)

ballots_numeric = pd.to_numeric(
    quality_df["analytical_ballots_cast"],
    errors="coerce"
)

comparable_ballots = (
    valid_votes_numeric.notna()
    & ballots_numeric.notna()
)

invalid_ballot_relationships = (
    valid_votes_numeric.loc[comparable_ballots]
    > ballots_numeric.loc[comparable_ballots]
).sum()

record_check(
    "Valid votes versus ballots cast",
    invalid_ballot_relationships == 0,
    f"Records where valid votes exceed ballots cast: "
    f"{invalid_ballot_relationships}"
)


# 10. Verify August 2017 exception statuses
august_exceptions = quality_df.loc[
    quality_df["election_id"].eq("2017_AUG_GENERAL")
    & quality_df["county"].isin(["Kisumu", "Turkana"]),
    ["county", "county_validation_status"]
]

expected_statuses = {
    "Kisumu": "INTERNAL_SOURCE_INCONSISTENCY",
    "Turkana": "UNRESOLVED_SOURCE_DIFFERENCE"
}

actual_statuses = dict(
    zip(
        august_exceptions["county"],
        august_exceptions["county_validation_status"]
    )
)

record_check(
    "August 2017 exception statuses",
    actual_statuses == expected_statuses,
    str(actual_statuses)
)


# 11. Verify 2022 remains county-source unverified
status_2022 = quality_df.loc[
    quality_df["election_id"].eq("2022_GENERAL"),
    "county_validation_status"
]

record_check(
    "2022 source verification status",
    status_2022.eq("COUNTY_SOURCE_UNVERIFIED").all(),
    "All 47 county records must remain unverified."
)


# Create and display the final assessment
quality_report = pd.DataFrame(quality_checks)

print("FINAL DATA-QUALITY ASSESSMENT")
print("=" * 65)

display(quality_report)

print("\nQUALITY CHECK SUMMARY")
print("Passed:", (quality_report["status"] == "PASS").sum())
print("Failed:", (quality_report["status"] == "FAIL").sum())


FINAL DATA-QUALITY ASSESSMENT


,check,status,details
0,Expected election records,PASS,Found 188 records; expected 188.
1,47 counties per election,PASS,"{'2013_GENERAL': 47, '2017_AUG_GENERAL': 47, '..."
2,Unique election-county records,PASS,Duplicate records: 0
3,Official county reconciliation,PASS,Matched official county records: 141/141
4,Registration completeness,PASS,Missing registration values: 0
5,Expected valid-vote exclusions,PASS,Missing valid-vote records: [('2017_AUG_GENERA...
6,Expected ballots-cast exclusions,PASS,Missing ballots-cast records: 49
7,Participation rate boundaries,PASS,All available calculated rates must be between...
8,Valid votes versus ballots cast,PASS,Records where valid votes exceed ballots cast: 0
9,August 2017 exception statuses,PASS,"{'Turkana': 'UNRESOLVED_SOURCE_DIFFERENCE', 'K..."



QUALITY CHECK SUMMARY
Passed: 11
Failed: 0


### Review the final analytical dataset structure
Before exporting to data/Processed/, I recommend reviewing the columns, completeness and election-level availability of the dataset.

This will help us distinguish fields suitable for analysis from source-specific fields retained for audit purposes.

In [109]:

# Cell 22: Review the integrated analytical dataset
# Diagnostic only — no files are created or modified.

final_review_df = participation_df.copy()

# 1. Review dataset dimensions and available columns
print("FINAL DATASET STRUCTURE")
print("=" * 65)

print("Rows:", len(final_review_df))
print("Columns:", len(final_review_df.columns))

print("\nCOLUMN NAMES AND DATA TYPES")

column_inventory = pd.DataFrame({
    "column_name": final_review_df.columns,
    "data_type": [
        str(dtype) for dtype in final_review_df.dtypes
    ],
    "non_null_records": final_review_df.notna().sum().values,
    "missing_records": final_review_df.isna().sum().values
})

display(column_inventory)


# 2. Review analytical measure availability by election
analytical_fields = [
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast",
    "calculated_turnout_pct",
    "valid_vote_participation_pct",
    "source_reported_turnout_pct"
]

print("\nANALYTICAL MEASURE AVAILABILITY BY ELECTION")

measure_availability = (
    final_review_df.groupby("election_id")[analytical_fields]
    .count()
    .reset_index()
)

display(measure_availability)


# 3. Review validation statuses
print("\nCOUNTY VALIDATION STATUS DISTRIBUTION")

validation_distribution = (
    final_review_df.groupby(
        ["election_id", "county_validation_status"],
        dropna=False
    )
    .size()
    .reset_index(name="county_count")
)

display(validation_distribution)


# 4. Preview selected analytical fields
print("\nANALYTICAL DATASET PREVIEW")

preview_columns = [
    "election_id",
    "county",
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast",
    "calculated_turnout_pct",
    "valid_vote_participation_pct",
    "source_reported_turnout_pct",
    "county_validation_status"
]

display(
    final_review_df[preview_columns].head(12)
)


FINAL DATASET STRUCTURE
Rows: 188
Columns: 34

COLUMN NAMES AND DATA TYPES


,column_name,data_type,non_null_records,missing_records
0,election_id,object,188,0
1,election_year,int64,188,0
2,election_date,object,188,0
3,election_type,object,188,0
4,county,object,188,0
5,elog_registered_voters,int64,188,0
6,elog_valid_votes,int64,188,0
7,elog_reported_turnout_pct,float64,188,0
8,winner,object,188,0
9,winner_votes,int64,188,0



ANALYTICAL MEASURE AVAILABILITY BY ELECTION


,election_id,analytical_registered_voters,analytical_valid_votes,analytical_ballots_cast,calculated_turnout_pct,valid_vote_participation_pct,source_reported_turnout_pct
0,2013_GENERAL,47,47,47,47,47,47
1,2017_AUG_GENERAL,47,45,45,45,45,47
2,2017_OCT_FRESH,47,47,47,47,47,47
3,2022_GENERAL,47,47,0,0,47,47



COUNTY VALIDATION STATUS DISTRIBUTION


,election_id,county_validation_status,county_count
0,2013_GENERAL,PARTIALLY_RECONCILED,47
1,2017_AUG_GENERAL,INTERNAL_SOURCE_INCONSISTENCY,1
2,2017_AUG_GENERAL,MATCHED,45
3,2017_AUG_GENERAL,UNRESOLVED_SOURCE_DIFFERENCE,1
4,2017_OCT_FRESH,MATCHED,47
5,2022_GENERAL,COUNTY_SOURCE_UNVERIFIED,47



ANALYTICAL DATASET PREVIEW


,election_id,county,analytical_registered_voters,analytical_valid_votes,analytical_ballots_cast,calculated_turnout_pct,valid_vote_participation_pct,source_reported_turnout_pct,county_validation_status
0,2013_GENERAL,Mombasa,413069.0,269314.0,272318.0,65.93,65.20,66.62,PARTIALLY_RECONCILED
1,2013_GENERAL,Kwale,175572.0,124626.0,125601.0,71.54,70.98,72.00,PARTIALLY_RECONCILED
2,2013_GENERAL,Kilifi,336410.0,215792.0,218174.0,64.85,64.15,64.91,PARTIALLY_RECONCILED
3,2013_GENERAL,Tana River,79641.0,64080.0,64589.0,81.10,80.46,81.29,PARTIALLY_RECONCILED
4,2013_GENERAL,Lamu,52359.0,43534.0,44171.0,84.36,83.15,84.38,PARTIALLY_RECONCILED
5,2013_GENERAL,Taita Taveta,114189.0,91333.0,92356.0,80.88,79.98,81.11,PARTIALLY_RECONCILED
6,2013_GENERAL,Garissa,115236.0,91382.0,91900.0,79.75,79.30,79.77,PARTIALLY_RECONCILED
7,2013_GENERAL,Wajir,118245.0,99695.0,100239.0,84.77,84.31,84.88,PARTIALLY_RECONCILED
8,2013_GENERAL,Mandera,120923.0,101271.0,101617.0,84.03,83.75,84.14,PARTIALLY_RECONCILED
9,2013_GENERAL,Marsabit,105259.0,89561.0,89882.0,85.39,85.09,85.92,PARTIALLY_RECONCILED


### Prepare the final analytical dataset
This cell creates a separate DataFrame with the proposed fields, converts numeric columns to appropriate types and adds a dedicated flag for the two turnout discrepancies.

It does not export any files yet.

In [112]:

# Cell 23: Prepare the final analytical dataset
# Create a separate reporting-ready DataFrame.
# No files are saved and participation_df is not modified.

final_columns = [
    "election_id",
    "election_year",
    "election_date",
    "election_type",
    "county",
    "county_key",
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast",
    "calculated_turnout_pct",
    "valid_vote_participation_pct",
    "source_reported_turnout_pct",
    "winner",
    "winner_votes",
    "runner_up",
    "margin",
    "county_validation_status",
    "turnout_verification_status",
    "validation_status"
]

# Check that every selected column exists
missing_columns = [
    col for col in final_columns
    if col not in participation_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Create a separate analytical dataset
final_analytical_df = participation_df[
    final_columns
].copy()


# Convert count fields to nullable integers
# Nullable Int64 preserves missing values without converting
# all counts to floating-point numbers.
count_columns = [
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast",
    "winner_votes",
    "margin"
]

for col in count_columns:
    numeric_values = pd.to_numeric(
        final_analytical_df[col],
        errors="raise"
    )

    # Counts must be whole numbers where present
    assert (
        numeric_values.dropna() % 1 == 0
    ).all(), f"Non-integer values found in {col}"

    final_analytical_df[col] = numeric_values.astype("Int64")


# Standardise percentage fields
percentage_columns = [
    "calculated_turnout_pct",
    "valid_vote_participation_pct",
    "source_reported_turnout_pct"
]

for col in percentage_columns:
    final_analytical_df[col] = pd.to_numeric(
        final_analytical_df[col],
        errors="raise"
    ).astype("float64")


# Add turnout-specific discrepancy status
# Keep it separate from the county vote-count validation status.
turnout_flag_keys = set(
    zip(
        turnout_audit["election_id"],
        turnout_audit["county"]
    )
)

final_analytical_df["turnout_discrepancy_flag"] = [
    (election_id, county) in turnout_flag_keys
    for election_id, county in zip(
        final_analytical_df["election_id"],
        final_analytical_df["county"]
    )
]


# Validate the new structure
assert len(final_analytical_df) == 188
assert final_analytical_df.shape[1] == 20
assert final_analytical_df["turnout_discrepancy_flag"].sum() == 2

assert not final_analytical_df.duplicated(
    ["election_id", "county_key"]
).any()


print("FINAL ANALYTICAL DATASET PREPARATION")
print("=" * 65)

print("Rows:", len(final_analytical_df))
print("Columns:", len(final_analytical_df.columns))

print("\nDATA TYPES")
display(final_analytical_df.dtypes.to_frame("data_type"))

print("\nTURNOUT DISCREPANCY FLAGS")
display(
    final_analytical_df.loc[
        final_analytical_df["turnout_discrepancy_flag"],
        [
            "election_id",
            "county",
            "calculated_turnout_pct",
            "source_reported_turnout_pct",
            "turnout_discrepancy_flag"
        ]
    ]
)

print("\nDATASET PREVIEW")
display(final_analytical_df.head(5))


FINAL ANALYTICAL DATASET PREPARATION
Rows: 188
Columns: 20

DATA TYPES


,data_type
election_id,object
election_year,int64
election_date,object
election_type,object
county,object
county_key,string[python]
analytical_registered_voters,Int64
analytical_valid_votes,Int64
analytical_ballots_cast,Int64
calculated_turnout_pct,float64



TURNOUT DISCREPANCY FLAGS


,election_id,county,calculated_turnout_pct,source_reported_turnout_pct,turnout_discrepancy_flag
66,2017_AUG_GENERAL,Kirinyaga,87.15,86.11,True
81,2017_AUG_GENERAL,Kericho,78.64,80.78,True



DATASET PREVIEW


,election_id,election_year,election_date,election_type,county,county_key,analytical_registered_voters,analytical_valid_votes,analytical_ballots_cast,calculated_turnout_pct,valid_vote_participation_pct,source_reported_turnout_pct,winner,winner_votes,runner_up,margin,county_validation_status,turnout_verification_status,validation_status,turnout_discrepancy_flag
0,2013_GENERAL,2013,2013-03-04,General Election,Mombasa,MOMBASA,413069,269314,272318,65.93,65.20,66.62,Raila Odinga,189985,Uhuru Kenyatta,125192,PARTIALLY_RECONCILED,IEBC_NUMBER_VOTED,PARTIALLY_RECONCILED,False
1,2013_GENERAL,2013,2013-03-04,General Election,Kwale,KWALE,175572,124626,125601,71.54,70.98,72.00,Raila Odinga,101407,Uhuru Kenyatta,83774,PARTIALLY_RECONCILED,IEBC_NUMBER_VOTED,PARTIALLY_RECONCILED,False
2,2013_GENERAL,2013,2013-03-04,General Election,Kilifi,KILIFI,336410,215792,218174,64.85,64.15,64.91,Raila Odinga,182708,Uhuru Kenyatta,159322,PARTIALLY_RECONCILED,IEBC_NUMBER_VOTED,PARTIALLY_RECONCILED,False
3,2013_GENERAL,2013,2013-03-04,General Election,Tana River,TANARIVER,79641,64080,64589,81.10,80.46,81.29,Raila Odinga,39666,Uhuru Kenyatta,17247,PARTIALLY_RECONCILED,IEBC_NUMBER_VOTED,PARTIALLY_RECONCILED,False
4,2013_GENERAL,2013,2013-03-04,General Election,Lamu,LAMU,52359,43534,44171,84.36,83.15,84.38,Raila Odinga,22962,Uhuru Kenyatta,5285,PARTIALLY_RECONCILED,IEBC_NUMBER_VOTED,PARTIALLY_RECONCILED,False


***One final consideration: Are the election-result fields internally consistent?***
Looking at the preview, the dataset includes winner, winner_votes, runner_up, and margin.

These fields originate from ELOG, whereas our participation measures have been reconciled against IEBC where possible.

We should therefore make one final check before export: do the winner-vote counts and margins remain numerically consistent with the analytical valid-vote totals?

For example, a leading candidate cannot have more votes than the total number of valid votes recorded in the same county-election event.

This is a useful integrity check, particularly because we have already identified source differences in August 2017.

### Validate election-result consistency

This is a diagnostic cell. It does not modify the analytical dataset.

In [118]:

# Cell 24: Validate presidential election-result consistency
# Diagnostic only — no datasets or files are modified.

result_review = final_analytical_df.copy()

# Ensure required numeric fields are available
result_fields = [
    "analytical_valid_votes",
    "winner_votes",
    "margin"
]

for col in result_fields:
    result_review[col] = pd.to_numeric(
        result_review[col],
        errors="raise"
    )


# 1. Check whether winner votes exceed total valid votes
# Exclude records with unresolved analytical valid-vote totals.
comparable_results = result_review.loc[
    result_review["analytical_valid_votes"].notna()
].copy()

winner_exceeds_valid = comparable_results.loc[
    comparable_results["winner_votes"]
    > comparable_results["analytical_valid_votes"]
]


# 2. Check for negative winner votes or margins
negative_winner_votes = result_review.loc[
    result_review["winner_votes"] < 0
]

negative_margins = result_review.loc[
    result_review["margin"] < 0
]


# 3. Check whether the reported margin exceeds winner votes
# A margin should not exceed the winner's own vote count.
margin_exceeds_winner = result_review.loc[
    result_review["margin"] > result_review["winner_votes"]
]


# 4. Review results by election
result_integrity_summary = (
    result_review.groupby("election_id")
    .agg(
        counties=("county", "count"),
        available_valid_votes=("analytical_valid_votes", "count"),
        available_winner_votes=("winner_votes", "count"),
        available_margins=("margin", "count")
    )
    .reset_index()
)

print("PRESIDENTIAL RESULT INTEGRITY SUMMARY")
print("=" * 65)
display(result_integrity_summary)

print("\nRESULT CONSISTENCY CHECKS")

result_checks = pd.DataFrame([
    {
        "check": "Winner votes exceed analytical valid votes",
        "flagged_records": len(winner_exceeds_valid)
    },
    {
        "check": "Negative winner votes",
        "flagged_records": len(negative_winner_votes)
    },
    {
        "check": "Negative margins",
        "flagged_records": len(negative_margins)
    },
    {
        "check": "Margins exceed winner votes",
        "flagged_records": len(margin_exceeds_winner)
    }
])

display(result_checks)


# 5. Show any records requiring investigation
flagged_indices = (
    winner_exceeds_valid.index
    .union(negative_winner_votes.index)
    .union(negative_margins.index)
    .union(margin_exceeds_winner.index)
)

print("\nFLAGGED COUNTY-ELECTION RECORDS")

if len(flagged_indices) == 0:
    print("No numerical inconsistencies identified.")
else:
    display(
        result_review.loc[
            flagged_indices,
            [
                "election_id",
                "county",
                "analytical_valid_votes",
                "winner",
                "winner_votes",
                "runner_up",
                "margin",
                "county_validation_status"
            ]
        ]
    )


PRESIDENTIAL RESULT INTEGRITY SUMMARY


,election_id,counties,available_valid_votes,available_winner_votes,available_margins
0,2013_GENERAL,47,47,47,47
1,2017_AUG_GENERAL,47,45,47,47
2,2017_OCT_FRESH,47,47,47,47
3,2022_GENERAL,47,47,47,47



RESULT CONSISTENCY CHECKS


,check,flagged_records
0,Winner votes exceed analytical valid votes,0
1,Negative winner votes,0
2,Negative margins,0
3,Margins exceed winner votes,0



FLAGGED COUNTY-ELECTION RECORDS
No numerical inconsistencies identified.


Cell 24 has passed all four presidential-result integrity checks. We can now proceed with the first controlled export of our Kenya Electoral Participation Analytics dataset.

The result is significant because we have checked not only voter registration and turnout calculations, but also the basic numerical consistency of the presidential election-result fields.

### Export the validated datasets

In [122]:

# Cell 25: Controlled export of the integrated election datasets
# Original raw data and existing reconciliation files are preserved.

from pathlib import Path
import pandas as pd

# Use the project directory established in earlier notebook cells.
# These paths are relative to the existing DOCS_DIR.
project_root = DOCS_DIR.parent

interim_dir = project_root / "data" / "interim"
processed_dir = project_root / "data" / "processed"

working_export_path = (
    interim_dir / "election_county_integrated_working.csv"
)

analytical_export_path = (
    processed_dir / "kenya_county_election_analytics.csv"
)

# Check that the expected project directories exist.
assert interim_dir.is_dir(), f"Missing directory: {interim_dir}"
assert processed_dir.is_dir(), f"Missing directory: {processed_dir}"

# Pre-export structural checks
assert participation_df.shape == (188, 34), (
    "Working dataset structure has changed."
)

assert final_analytical_df.shape == (188, 20), (
    "Analytical dataset structure has changed."
)

assert not final_analytical_df.duplicated(
    ["election_id", "county_key"]
).any(), "Duplicate election-county records detected."

assert (
    quality_report["status"].eq("PASS").all()
), "One or more final quality checks failed."

assert (
    result_checks["flagged_records"].eq(0).all()
), "Presidential-result integrity checks have flagged records."

# Confirm that the documented exceptions are preserved.
assert final_analytical_df[
    "analytical_valid_votes"
].isna().sum() == 2

assert final_analytical_df[
    "analytical_ballots_cast"
].isna().sum() == 49

assert final_analytical_df[
    "turnout_discrepancy_flag"
].sum() == 2

# Avoid silently overwriting an existing export.
assert not working_export_path.exists(), (
    f"Working export already exists: {working_export_path}"
)

assert not analytical_export_path.exists(), (
    f"Analytical export already exists: {analytical_export_path}"
)

# Save both datasets.
participation_df.to_csv(
    working_export_path,
    index=False
)

final_analytical_df.to_csv(
    analytical_export_path,
    index=False
)

# Read back and verify the exported files.
saved_working = pd.read_csv(working_export_path)
saved_analytical = pd.read_csv(analytical_export_path)

assert saved_working.shape == participation_df.shape
assert saved_analytical.shape == final_analytical_df.shape

assert list(saved_analytical.columns) == list(
    final_analytical_df.columns
)

assert not saved_analytical.duplicated(
    ["election_id", "county_key"]
).any()

assert saved_analytical[
    "analytical_valid_votes"
].isna().sum() == 2

assert saved_analytical[
    "analytical_ballots_cast"
].isna().sum() == 49

assert saved_analytical[
    "turnout_discrepancy_flag"
].sum() == 2

print("CONTROLLED DATASET EXPORT COMPLETE")
print("=" * 65)

print("\nWorking dataset")
print("Path:", working_export_path)
print("Shape:", saved_working.shape)

print("\nAnalytical dataset")
print("Path:", analytical_export_path)
print("Shape:", saved_analytical.shape)

print("\nEXPORTED ANALYTICAL RECORDS BY ELECTION")
display(
    saved_analytical.groupby("election_id")
    .size()
    .reset_index(name="county_records")
)

print("\nExport verification: PASSED")


CONTROLLED DATASET EXPORT COMPLETE

Working dataset
Path: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim\election_county_integrated_working.csv
Shape: (188, 34)

Analytical dataset
Path: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\processed\kenya_county_election_analytics.csv
Shape: (188, 20)

EXPORTED ANALYTICAL RECORDS BY ELECTION


,election_id,county_records
0,2013_GENERAL,47
1,2017_AUG_GENERAL,47
2,2017_OCT_FRESH,47
3,2022_GENERAL,47



Export verification: PASSED


***Save the methodology and validation summary***

Before closing Notebook 03, let's document the decisions that produced the analytical dataset. This is particularly important because the data contains known source limitations that must accompany any future analysis.

The following cell creates a Markdown methodology document in your existing documentation/ folder. It does not modify either exported dataset.

In [126]:

# Cell 26: Create methodology and validation documentation
# Documentation only — exported datasets remain unchanged.

from pathlib import Path

methodology_path = (
    DOCS_DIR / "election_data_methodology_and_validation.md"
)

methodology_text = """# Kenya Electoral Participation Analytics
## Data Methodology and Validation Summary

### 1. Project scope

This dataset contains county-level presidential election
information for Kenya's 47 counties across four election events:

- 2013 General Election
- August 2017 General Election
- October 2017 Fresh Presidential Election
- 2022 General Election

The October 2017 election is treated as a distinct electoral
event rather than a separate general election.

### 2. Data sources

The project uses election information extracted from
ELOG reports and official IEBC election documents.

ELOG data provides the initial county-level election records.

IEBC documents provide official county-level figures used
for reconciliation where available.

The availability and completeness of official county-level
figures differ across election events.

### 3. Analytical measures

**Registered voters**

The selected registration denominator for each county
and election event.

**Valid votes**

The selected count of valid presidential votes.

**Ballots cast**

The selected count of ballots cast, including rejected
ballots where those components are available.

**Calculated turnout (%)**

Calculated as:

    (Analytical ballots cast / Analytical registered voters) * 100

**Valid-vote participation (%)**

Calculated as:

    (Analytical valid votes / Analytical registered voters) * 100

**Source-reported turnout (%)**

The turnout percentage reported in the ELOG source.

This is retained separately from calculated turnout.

### 4. Election-specific reconciliation decisions

#### 2013 General Election

- Registration uses the IEBC total registered-voter count,
  including biometric and non-biometric registration.
- Analytical valid votes are retained from ELOG.
- Ballots cast use the IEBC number-voted field.
- All 47 county records are classified as partially reconciled.
- Differences in turnout definitions and denominators
  require care when comparing source-reported percentages.

#### August 2017 General Election

- Registration uses IEBC county figures.
- Valid votes and rejected ballots use IEBC county figures
  where the source reconciliation is acceptable.
- Ballots cast are calculated from valid votes plus
  rejected ballots for reconciled records.
- 45 counties have matched vote-count records.
- Kisumu is flagged for an internal inconsistency in
  the printed IEBC source.
- Turkana is flagged for an unresolved difference
  between ELOG and IEBC valid-vote figures.
- Analytical valid votes, ballots cast and calculated
  turnout remain missing for Kisumu and Turkana.

Two additional turnout discrepancies are documented:

- Kirinyaga: calculated 87.15%; ELOG reported 86.11%.
- Kericho: calculated 78.64%; ELOG reported 80.78%.

These are turnout-specific discrepancies and do not
replace the vote-count reconciliation statuses.

#### October 2017 Fresh Presidential Election

- Registration and valid-vote counts match the available
  official county figures for all 47 counties.
- Ballots cast are calculated from IEBC valid votes
  and rejected ballots.
- The election is analysed as a distinct event.

#### 2022 General Election

- County registration and valid-vote figures are
  retained from ELOG.
- Independent reconciliation against official
  county-level results has not been completed.
- Analytical ballots cast and calculated turnout
  remain unavailable for all 47 counties.
- Valid-vote participation is available but relies
  on the provisional county figures.
- ELOG-reported turnout is retained as a separate field.

### 5. Presidential candidate results

Winner, winner-vote, runner-up and margin fields
are derived from ELOG.

Basic numerical integrity checks were completed:

- Winner votes do not exceed available analytical valid votes.
- Winner-vote counts are nonnegative.
- Margins are nonnegative.
- Margins do not exceed winner votes.

These checks do not constitute independent candidate-level
reconciliation against official IEBC results.

### 6. Validation outcomes

- Total election-county records: 188
- Counties per election event: 47
- Official county records successfully joined: 141
- Duplicate election-county records: 0
- Final structural and participation checks passed: 11 of 11
- Presidential-result integrity checks flagged: 0 records
- August 2017 unresolved valid-vote records: 2
- August 2017 turnout discrepancies above 1 pp: 2
- Missing analytical ballots-cast values: 49

Passing these checks establishes consistency with the
documented analytical rules. It does not independently
verify every underlying source figure.

### 7. Dataset outputs

**Working dataset**

    data/Interim/election_county_integrated_working.csv

Contains 188 records and 34 fields, including supporting
source and reconciliation information.

**Processed analytical dataset**

    data/Processed/kenya_county_election_analytics.csv

Contains 188 records and 20 fields selected for analysis.

### 8. Analytical limitations

- Calculated turnout and valid-vote participation are
  different measures and should not be interchanged.
- August and October 2017 must remain distinct events.
- The two unresolved August 2017 county records should
  not be silently included in aggregate calculations
  requiring missing analytical values.
- The 2022 county data remains independently unverified.
- Source-reported 2022 turnout should not be presented
  as independently calculated turnout.
- Comparisons across election years must acknowledge
  differences in source coverage and methodology.

### 9. Reproducibility

The project preserves original raw source files,
intermediate extracted datasets, reconciliation registers,
turnout discrepancy documentation and analytical outputs.

Source discrepancies are documented rather than silently
corrected.
"""

# Protect an existing methodology document.
assert not methodology_path.exists(), (
    f"Methodology file already exists: {methodology_path}"
)

methodology_path.write_text(
    methodology_text,
    encoding="utf-8"
)

# Verify the saved document.
saved_methodology = methodology_path.read_text(
    encoding="utf-8"
)

assert saved_methodology == methodology_text
assert methodology_path.is_file()

print("METHODOLOGY DOCUMENTATION SAVED")
print("=" * 65)
print("File:", methodology_path)
print("Characters:", len(saved_methodology))
print("Verification: PASSED")


METHODOLOGY DOCUMENTATION SAVED
File: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\election_data_methodology_and_validation.md
Characters: 5802
Verification: PASSED


Cell 26 has passed successfully. This officially completes Notebook 03 — Election Data Preparation.

We now have a validated analytical dataset, preserved reconciliation records, and a methodology document explaining our data sources, calculations, exceptions and limitations.